# Arrays y Listas
- Un tipo de colección es aquel que puede almacenar múltiples valores. La lista y la tupla son 2 ejemplos de Python.
- Arrays (`pl.Array`) y listas (`pl.List`) son dos tipos de colección en Polars.
- Ciertas expresiones de Polars devuelven una columna donde cada fila almacena un array/lista.
- El diseño permite que cada fila almacene una secuencia ordenada de valores, una colección de valores.
- Polars usará arrays cuando cada fila tenga la misma cantidad de valores dentro de la colección.
- Polars usará listas cuando las filas tengan una cantidad diferente de valores dentro de la colección.

In [2]:
import polars as pl

### Listas de Python vs. Listas de Polars
- Las listas de Python pueden almacenar datos heterogéneos (valores de diferentes tipos).
- Las listas/arrays de Polars almacenan datos homogéneos (los valores deben ser del mismo tipo).

In [ ]:
[1, False, "hello", 4.14]

[1, False, 'hello', 4.14]

- Construyamos un `DataFrame` que almacene una columna de listas.
- Pasaremos al constructor `pl.DataFrame` un diccionario que mapea nombres de columnas a valores.
- Pasamos una lista de listas para los valores de la columna.
- Polars asume un tipo `pl.List` por defecto. Este es un tipo distinto de Polars.
- Polars inferirá el tipo de datos de los elementos de la lista.
- Observa que el tipo de datos de los valores de la columna `names` es una lista que almacena cadenas (`list[str]`).

In [4]:
pl.DataFrame(
    {"names": [["Paul", "Molly"], ["John", "Jenna", "Jim"], ["Shenny", "Pauline"], []]}
)

names
list[str]
"[""Paul"", ""Molly""]"
"[""John"", ""Jenna"", ""Jim""]"
"[""Shenny"", ""Pauline""]"
[]


- Podemos asignar explícitamente el tipo de dato de Polars (`pl.List`) a la columna `names`.
- El parámetro `schema` acepta un diccionario con un mapeo completo de columnas a sus tipos correspondientes.
- El parámetro `schema_overrides` acepta un diccionario que proporciona los tipos de datos de columna a _sobrescribir_.
- Los parámetros `schema` y `schema_overrides` son equivalentes aquí porque solo hay una columna de datos.

In [ ]:
pl.DataFrame(
    {"names": [["Paul", "Molly"], ["John", "Jenna", "Jim"], ["Shenny", "Pauline"], []]},
    schema_overrides={"names": pl.List},
)

names
list[str]
"[""Paul"", ""Molly""]"
"[""John"", ""Jenna"", ""Jim""]"
"[""Shenny"", ""Pauline""]"
[]


- `pl.List` es un constructor por sí mismo. Podemos pasarle el tipo de dato explícito de los elementos de la lista.

In [ ]:
pl.DataFrame(
    {"names": [["Paul", "Molly"], ["John", "Jenna", "Jim"], ["Shenny", "Pauline"], []]},
    schema_overrides={"names": pl.List(pl.String)},
)

names
list[str]
"[""Paul"", ""Molly""]"
"[""John"", ""Jenna"", ""Jim""]"
"[""Shenny"", ""Pauline""]"
[]


- La sintaxis de `pl.List` es útil cuando se desea sobrescribir los valores predeterminados de Polars.
- Por ejemplo, Polars inferirá un tipo entero predeterminado de `i64` para una columna de listas de enteros.
- Usa `pl.List` para cambiar el tipo entero `i64` a otro tipo entero.

In [8]:
# pl.DataFrame({"values": [[1, 2, 3]]})

In [7]:
pl.DataFrame({"values": [[1, 2, 3]]}, schema_overrides={"values": pl.List(pl.Int8)})

values
list[i8]
"[1, 2, 3]"


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/lists-and-arrays/#the-data-type-list
- https://docs.pola.rs/api/python/stable/reference/api/polars.datatypes.List.html#polars.datatypes.List

## El Método str.split
- A menudo llegarás al tipo de dato `pl.List` a través de una transformación.
- Imagina que estás registrando los invitados a una cena.
- Los invitados llegan al programa Python como una lista de cadenas.
- Cada cadena representa un grupo de personas en una mesa.

In [9]:
guests = pl.DataFrame({"names": ["Paul,Molly", "John,Jenna,Jim", "Shenny,Pauline"]})
guests

names
str
"""Paul,Molly"""
"""John,Jenna,Jim"""
"""Shenny,Pauline"""


- El espacio de nombres/atributo `str` contiene métodos para operaciones/expresiones con cadenas.
- El método `str.split` aplica la lógica de `split` a cada valor de fila.
- El método `str.split` devuelve una columna de listas.

In [10]:
guests.select(pl.col("names").str.split(","))

names
list[str]
"[""Paul"", ""Molly""]"
"[""John"", ""Jenna"", ""Jim""]"
"[""Shenny"", ""Pauline""]"


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.str.split.html

## El atributo `list`
- Polars define las funcionalidades de lista dentro de un atributo/espacio de nombres `list`.
- El método `len` devuelve la longitud de cada lista.
- El método `head` extrae los primeros `n` elementos de cada lista. Devuelve una columna de listas.
- El método `tail` extrae los últimos `n` elementos de cada lista. Devuelve una columna de listas.
- El método `first` extrae el primer elemento de cada lista.
- El método `last` extrae el último elemento de cada lista.

In [11]:
guests = pl.DataFrame(
    {"names": [["Paul", "Molly"], ["John", "Jenna", "Jim"], ["Shenny", "Pauline"], []]},
)
guests

names
list[str]
"[""Paul"", ""Molly""]"
"[""John"", ""Jenna"", ""Jim""]"
"[""Shenny"", ""Pauline""]"
[]


In [12]:
guests.with_columns(
    pl.col("names").list.len().alias("length"),
    pl.col("names").list.head(2).alias("head"),
    pl.col("names").list.tail(2).alias("tail"),
    pl.col("names").list.first().alias("first"),
    pl.col("names").list.last().alias("last"),
)

names,length,head,tail,first,last
list[str],u32,list[str],list[str],str,str
"[""Paul"", ""Molly""]",2,"[""Paul"", ""Molly""]","[""Paul"", ""Molly""]","""Paul""","""Molly"""
"[""John"", ""Jenna"", ""Jim""]",3,"[""John"", ""Jenna""]","[""Jenna"", ""Jim""]","""John""","""Jim"""
"[""Shenny"", ""Pauline""]",2,"[""Shenny"", ""Pauline""]","[""Shenny"", ""Pauline""]","""Shenny""","""Pauline"""
[],0,[],[],null,null


- El método `contains` verifica la inclusión de un elemento dentro de la lista de cada fila.
- Pasa una expresión booleana al método `filter` para filtrar filas.

In [14]:
guests.with_columns(pl.col("names").list.contains("Jenna").alias("has_jenna"))

names,has_jenna
list[str],bool
"[""Paul"", ""Molly""]",false
"[""John"", ""Jenna"", ""Jim""]",true
"[""Shenny"", ""Pauline""]",false
[],false


In [15]:
guests.filter(pl.col("names").list.contains("Jenna").alias("has_jenna"))

names
list[str]
"[""John"", ""Jenna"", ""Jim""]"


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/lists-and-arrays/#the-namespace-list
- https://docs.pola.rs/user-guide/expressions/lists-and-arrays/#operating-on-lists
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.len.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.head.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.tail.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.first.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.last.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.contains.html

## Ordenar las Listas
En Polars, cuando ordenas una columna de tipo List usando `.sort()`, se comparan los elementos de las listas posición por posición (orden lexicográfico):

- **Comparación por elementos:** *Compara el primer elemento de cada lista*. En el ejemplo, una lista vacía [] se considera menor que cualquier lista con contenido, por lo que aparece primero.
- **Orden alfabético:** Luego se comparan las cadenas. 'John' (J) viene antes que 'Paul' (P), y 'Paul' viene antes que 'Shenny' (S).



In [16]:
guests = pl.DataFrame(
    {"names": [["Paul", "Molly"], ["John", "Jenna", "Jim"], ["Shenny", "Pauline"], []]},
)
guests

names
list[str]
"[""Paul"", ""Molly""]"
"[""John"", ""Jenna"", ""Jim""]"
"[""Shenny"", ""Pauline""]"
[]


In [17]:
guests.sort("names")

names
list[str]
[]
"[""John"", ""Jenna"", ""Jim""]"
"[""Paul"", ""Molly""]"
"[""Shenny"", ""Pauline""]"


- El método `list.sort` ordena los elementos de cada lista. El orden de las filas en la columna permanece igual.
- Polars se adapta a listas de diferentes longitudes.

In [18]:
guests.with_columns(pl.col("names").list.sort().alias("sorted_names"))

names,sorted_names
list[str],list[str]
"[""Paul"", ""Molly""]","[""Molly"", ""Paul""]"
"[""John"", ""Jenna"", ""Jim""]","[""Jenna"", ""Jim"", ""John""]"
"[""Shenny"", ""Pauline""]","[""Pauline"", ""Shenny""]"
[],[]


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.sort.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.sort.html

## El Método `explode`
- El método `list.explode` es una operación de "aplanamiento". Crea una secuencia unidimensional de valores a partir de una colección de listas.


In [19]:
# Funcionaría con with_columns?
guests = pl.DataFrame(
    {"names": [["Molly", "Paul"], ["John", "Jenna", "Jim"], ["Shenny", "Pauline"], []]}
)
guests

names
list[str]
"[""Molly"", ""Paul""]"
"[""John"", ""Jenna"", ""Jim""]"
"[""Shenny"", ""Pauline""]"
[]


- El método `list.explode` devuelve una columna más larga que `names` (una fila por nombre).
- La columna `names` tiene 4 filas pero hay 7 nombres en total para extraer.
- El método `select` devuelve una nueva columna que puede ser de cualquier longitud.
- Una lista vacía se evalúa como un único valor nulo.

In [20]:
guests.select(pl.col("names").list.explode())

names
str
"""Molly"""
"""Paul"""
"""John"""
"""Jenna"""
"""Jim"""
"""Shenny"""
"""Pauline"""
null


- El método `flatten` logra el mismo resultado.

In [ ]:
# guests.select(pl.col("names").flatten())

- También podemos invocar `explode` sobre el `DataFrame` y pasarle una expresión de columna.

In [ ]:
names = guests.explode(pl.col("names"))
names

names
str
"""Molly"""
"""Paul"""
"""John"""
"""Jenna"""
"""Jim"""
"""Shenny"""
"""Pauline"""
null


In [ ]:
names.drop_nulls(pl.col("names"))

names
str
"""Molly"""
"""Paul"""
"""John"""
"""Jenna"""
"""Jim"""
"""Shenny"""
"""Pauline"""


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.explode.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.drop_nulls.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.flatten.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.explode.html

## Explode con Múltiples Columnas de Listas
- Digamos que estamos mapeando invitados de un evento a los platos principales entregados a su mesa.

In [21]:
guests = pl.DataFrame(
    {
        "names": [
            ["Paul", "Molly"],
            ["John", "Jenna", "Jim"],
            ["Shenny", "Pauline"],
            [],
        ],
        "entrees": [
            ["Fish", "Steak"],
            ["Fish", "Chicken", "Steak"],
            ["Chicken", "Steak"],
            [],
        ],
    },
)

guests

names,entrees
list[str],list[str]
"[""Paul"", ""Molly""]","[""Fish"", ""Steak""]"
"[""John"", ""Jenna"", ""Jim""]","[""Fish"", ""Chicken"", ""Steak""]"
"[""Shenny"", ""Pauline""]","[""Chicken"", ""Steak""]"
[],[]


In [22]:
guests.explode("names", "entrees")

names,entrees
str,str
"""Paul""","""Fish"""
"""Molly""","""Steak"""
"""John""","""Fish"""
"""Jenna""","""Chicken"""
"""Jim""","""Steak"""
"""Shenny""","""Chicken"""
"""Pauline""","""Steak"""
null,null


- La lista de platos principales puede ser más larga o más corta que el número de invitados.


In [23]:
guests = pl.DataFrame(
    {
        "names": [
            ["Paul", "Molly"],
            ["John", "Jenna", "Jim"],
            ["Shenny", "Pauline"],
            [],
        ],
        "entrees": [
            ["Fish", "Steak"],
            ["Fish", "Chicken"],
            ["Chicken", "Steak", "Barbecue"],
            [],
        ],
    },
)

guests

names,entrees
list[str],list[str]
"[""Paul"", ""Molly""]","[""Fish"", ""Steak""]"
"[""John"", ""Jenna"", ""Jim""]","[""Fish"", ""Chicken""]"
"[""Shenny"", ""Pauline""]","[""Chicken"", ""Steak"", ""Barbecue""]"
[],[]


In [25]:
# Funcionará con with_columns
# guests.explode("names", "entrees")

- Podemos hacer explode primero en la columna `names` para obtener cada combinación de persona con los platos que ordenaron.
- Polars empareja cada nombre dentro de la lista de una fila con la lista completa de `entrees` que recibieron.

In [26]:
guests.explode("names")

names,entrees
str,list[str]
"""Paul""","[""Fish"", ""Steak""]"
"""Molly""","[""Fish"", ""Steak""]"
"""John""","[""Fish"", ""Chicken""]"
"""Jenna""","[""Fish"", ""Chicken""]"
"""Jim""","[""Fish"", ""Chicken""]"
"""Shenny""","[""Chicken"", ""Steak"", ""Barbecue""]"
"""Pauline""","[""Chicken"", ""Steak"", ""Barbecue""]"
null,[]


- Ahora, podemos hacer explode en la lista `entrees` para crear una fila por cada combinación de invitado y cada comida en la lista de platos principales.

In [27]:
guests.explode("names").explode("entrees")

names,entrees
str,str
"""Paul""","""Fish"""
"""Paul""","""Steak"""
"""Molly""","""Fish"""
"""Molly""","""Steak"""
"""John""","""Fish"""
…,…
"""Shenny""","""Barbecue"""
"""Pauline""","""Chicken"""
"""Pauline""","""Steak"""


- Invertir el orden de las llamadas al método `explode` produce el mismo `DataFrame`.

In [ ]:
guests.explode("entrees").explode("names").drop_nulls()

names,entrees
str,str
"""Paul""","""Fish"""
"""Molly""","""Fish"""
"""Paul""","""Steak"""
"""Molly""","""Steak"""
"""John""","""Fish"""
…,…
"""Pauline""","""Chicken"""
"""Shenny""","""Steak"""
"""Pauline""","""Steak"""


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.explode.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.drop_nulls.html

## Operaciones Matemáticas
- El atributo `list` incluye varios métodos matemáticos.
- Polars invocará la operación en cada fila/lista dentro de una columna.
- Polars puede truncar la visualización de elementos de la lista si esta es larga.
- Los puntos suspensivos (`...`) marcan un hueco en la lista.

In [28]:
work = pl.DataFrame(
    {
        "employee": ["Alice", "Bob", "Carol", "Dave"],
        "questions_per_meeting": [[6, 10, 8], [7, 8, 1, 1], [4, 4], []],
    }
)

work

employee,questions_per_meeting
str,list[i64]
"""Alice""","[6, 10, 8]"
"""Bob""","[7, 8, … 1]"
"""Carol""","[4, 4]"
"""Dave""",[]


- El método `list.sum` suma los valores de cada lista.
- Los métodos `list.max` y `list.min` extraen el valor más grande y más pequeño de cada lista.
- El método `list.median` devuelve el punto medio de cada lista cuando se ordena.
- El método `list.n_unique` devuelve la cantidad de elementos únicos dentro de cada lista.

In [29]:
work.with_columns(
    pl.col("questions_per_meeting").list.sum().alias("sum"),
    pl.col("questions_per_meeting").list.max().alias("max"),
    pl.col("questions_per_meeting").list.min().alias("min"),
    pl.col("questions_per_meeting").list.mean().alias("mean"),
    pl.col("questions_per_meeting").list.median().alias("median"),
    pl.col("questions_per_meeting").list.n_unique().alias("n_unique"),
)

employee,questions_per_meeting,sum,max,min,mean,median,n_unique
str,list[i64],i64,i64,i64,f64,f64,u32
"""Alice""","[6, 10, 8]",24,10,6,8.0,8.0,3
"""Bob""","[7, 8, … 1]",17,8,1,4.25,4.0,3
"""Carol""","[4, 4]",8,4,4,4.0,4.0,1
"""Dave""",[],0,null,null,null,null,0


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.sum.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.max.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.min.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.mean.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.median.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.n_unique.html

## Los Métodos `list.eval`, `list.any` y `list.all`
- El método `list.eval` mapea cada elemento de la lista a un nuevo valor.
- El método `list.eval` devuelve una lista con las evaluaciones.

In [30]:
work = pl.DataFrame(
    {
        "employee": ["Alice", "Bob", "Carol", "Dave"],
        "questions_per_meeting": [[6, 10, 8], [7, 8, 1, 1], [4, 4], []],
    }
)
work

employee,questions_per_meeting
str,list[i64]
"""Alice""","[6, 10, 8]"
"""Bob""","[7, 8, … 1]"
"""Carol""","[4, 4]"
"""Dave""",[]


- La función `pl.element` devuelve cada elemento de la lista uno por uno.
- La función `pl.element` está diseñada para usarse con el método `eval`.

In [31]:
work.with_columns(
    pl.col("questions_per_meeting").list.eval(pl.element() + 1)
    .alias("new_column")
)

employee,questions_per_meeting,new_column
str,list[i64],list[i64]
"""Alice""","[6, 10, 8]","[7, 11, 9]"
"""Bob""","[7, 8, … 1]","[8, 9, … 2]"
"""Carol""","[4, 4]","[5, 5]"
"""Dave""",[],[]


In [32]:
tough_meetings = work.with_columns(
    pl.col("questions_per_meeting")
    .list.eval(pl.element() > 5)
    .alias("tough_meeting")
)
tough_meetings

employee,questions_per_meeting,tough_meeting
str,list[i64],list[bool]
"""Alice""","[6, 10, 8]","[true, true, true]"
"""Bob""","[7, 8, … 1]","[true, true, … false]"
"""Carol""","[4, 4]","[false, false]"
"""Dave""",[],[]


- El método `list.eval` nos dio convenientemente una columna de listas booleanas.
- El método `list.any` devuelve True si una lista contiene al menos un valor True.
- El método `list.all` devuelve True si todos los valores dentro de la lista son True.

In [34]:
tough_meetings.with_columns(
    pl.col("tough_meeting").list.any().alias("at_least_1_tough_meeting"),
    pl.col("tough_meeting").list.all().alias("all_tough_meetings"),
)

employee,questions_per_meeting,tough_meeting,at_least_1_tough_meeting,all_tough_meetings
str,list[i64],list[bool],bool,bool
"""Alice""","[6, 10, 8]","[true, true, true]",true,true
"""Bob""","[7, 8, … 1]","[true, true, … false]",true,false
"""Carol""","[4, 4]","[false, false]",false,false
"""Dave""",[],[],false,true


In [35]:
tough_meetings = work.with_columns(
    pl.col("questions_per_meeting")
    .list.eval(pl.element().filter(pl.element() > 5))
    .alias("tough_meeting")
)
tough_meetings

employee,questions_per_meeting,tough_meeting
str,list[i64],list[i64]
"""Alice""","[6, 10, 8]","[6, 10, 8]"
"""Bob""","[7, 8, … 1]","[7, 8]"
"""Carol""","[4, 4]",[]
"""Dave""",[],[]


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/lists-and-arrays/#operating-on-lists
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.eval.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.any.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.all.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.element.html

## Concatenar Valores de Columnas

In [36]:
import polars as pl

In [37]:
action_stars = pl.DataFrame(
    {
        "first_name": ["Arnold", "Sylvester", "Jean-Claude"],
        "last_name": ["Schwarzenegger", "Stallone", "Van Damme"],
    }
)

action_stars

first_name,last_name
str,str
"""Arnold""","""Schwarzenegger"""
"""Sylvester""","""Stallone"""
"""Jean-Claude""","""Van Damme"""


- La función `format` permite la interpolación de múltiples valores de columna dentro de la cadena formateada.

In [39]:
# Similar a los f-string de Python
action_stars.with_columns(
    pl.format(
        "{} {} the Great", pl.col("first_name").str.to_uppercase(), pl.col("last_name")
    ).alias("full_name")
)

first_name,last_name,full_name
str,str,str
"""Arnold""","""Schwarzenegger""","""ARNOLD Schwarzenegger the Grea…"
"""Sylvester""","""Stallone""","""SYLVESTER Stallone the Great"""
"""Jean-Claude""","""Van Damme""","""JEAN-CLAUDE Van Damme the Grea…"


- La función `pl.concat_list` concatena contenido entre columnas en una lista en su lugar.

In [41]:
action_stars_with_name_list = action_stars.with_columns(
    pl.concat_list(pl.col("first_name"), pl.col("last_name")).alias("name_list")
)
action_stars_with_name_list

first_name,last_name,name_list
str,str,list[str]
"""Arnold""","""Schwarzenegger""","[""Arnold"", ""Schwarzenegger""]"
"""Sylvester""","""Stallone""","[""Sylvester"", ""Stallone""]"
"""Jean-Claude""","""Van Damme""","[""Jean-Claude"", ""Van Damme""]"


- Supongamos que tenemos la situación inversa: tenemos una columna de listas y queremos concatenar el contenido en una nueva columna.

In [43]:
action_stars_with_name_list.select(pl.col("name_list").list.join(separator=", ").alias("full_name"))

full_name
str
"""Arnold,Schwarzenegger"""
"""Sylvester,Stallone"""
"""Jean-Claude,Van Damme"""


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/lists-and-arrays/#row-wise-computations
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.format.html#polars.format
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.concat_str.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.concat_list.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.list.join.html

## Arrays
- El array de Polars es un tipo de almacenamiento complementario para colecciones.
- Una columna de listas puede tener una longitud de lista diferente en cada fila.
- Una columna de arrays debe tener la misma longitud de array en cada fila.
- La consistencia en longitud permite que el Array sea más eficiente en memoria y rendimiento.
- La documentación de Polars recomienda usar el Array sobre la Lista _si es posible_.
- Al instanciar un `DataFrame`, Polars asumirá un tipo lista incluso con entradas de colección de igual tamaño.

In [44]:
pl.DataFrame(
    {
        "burritos": [
            ["white rice", "pinto beans", "steak"],
            ["brown rice", "black beans", "chicken"],
        ]
    }
)

burritos
list[str]
"[""white rice"", ""pinto beans"", ""steak""]"
"[""brown rice"", ""black beans"", ""chicken""]"


- Usa los parámetros `schema` o `schema_overrides` para sobrescribir el tipo inferido de una columna.
- El constructor `pl.Array` acepta dos argumentos: el tipo de cada elemento del array y la longitud de cada array.

In [45]:
lunch = pl.DataFrame(
    {
        "burritos": [
            ["white rice", "pinto beans", "steak"],
            ["brown rice", "black beans", "chicken"],
        ]
    },
    schema_overrides={"burritos": pl.Array(pl.String, shape=3)},
)
lunch

burritos
"array[str, 3]"
"[""white rice"", ""pinto beans"", ""steak""]"
"[""brown rice"", ""black beans"", ""chicken""]"


In [ ]:
lunch.schema

Schema([('burritos', Array(String, shape=(3,)))])

- Expandamos el `DataFrame` para incluir una columna de arrays de punto flotante.

In [46]:
lunch = pl.DataFrame(
    {
        "burritos": [
            ["white rice", "pinto beans", "steak"],
            ["brown rice", "black beans", "chicken"],
        ],
        "calories": [[205, 245, 349], [218, 227, 215]],
    },
    schema_overrides={
        "burritos": pl.Array(pl.String, shape=3),
        "calories": pl.Array(pl.UInt16, shape=3),
    },
)
lunch

burritos,calories
"array[str, 3]","array[u16, 3]"
"[""white rice"", ""pinto beans"", ""steak""]","[205, 245, 349]"
"[""brown rice"", ""black beans"", ""chicken""]","[218, 227, 215]"


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/lists-and-arrays/#the-data-type-array
- https://docs.pola.rs/user-guide/expressions/lists-and-arrays/#creating-an-array-column
- https://docs.pola.rs/api/python/stable/reference/api/polars.datatypes.Array.html#polars.datatypes.Array

## El Atributo `arr`
- El array soporta una colección de métodos similar a la lista.
- Polars anida los métodos bajo el atributo/espacio de nombres `arr`.

In [ ]:
lunch = pl.DataFrame(
    {
        "burritos": [
            ["white rice", "pinto beans", "steak"],
            ["brown rice", "black beans", "chicken"],
        ],
        "calories": [[205, 245, 349], [218, 227, 215]],
    },
    schema_overrides={
        "burritos": pl.Array(pl.String, shape=3),
        "calories": pl.Array(pl.Int32, shape=3),
    },
)

lunch

burritos,calories
"array[str, 3]","array[i32, 3]"
"[""white rice"", ""pinto beans"", ""steak""]","[205, 245, 349]"
"[""brown rice"", ""black beans"", ""chicken""]","[218, 227, 215]"


In [47]:
lunch.with_columns(
    pl.col("calories").arr.sum().alias("calorie_sum"),
    pl.col("calories").arr.mean().alias("calorie_average"),
    pl.col("calories").arr.max().alias("largest"),
    pl.col("calories").arr.min().alias("smallest"),
    pl.col("burritos").arr.head(2).alias("first_two"),
    pl.col("burritos").arr.tail(2).alias("last_two"),
    pl.col("burritos").arr.first().alias("first"),
    pl.col("burritos").arr.last().alias("last"),
)

burritos,calories,calorie_sum,calorie_average,largest,smallest,first_two,last_two,first,last
"array[str, 3]","array[u16, 3]",i64,f64,u16,u16,list[str],list[str],str,str
"[""white rice"", ""pinto beans"", ""steak""]","[205, 245, 349]",799,266.333333,349,205,"[""white rice"", ""pinto beans""]","[""pinto beans"", ""steak""]","""white rice""","""steak"""
"[""brown rice"", ""black beans"", ""chicken""]","[218, 227, 215]",660,220.0,227,215,"[""brown rice"", ""black beans""]","[""black beans"", ""chicken""]","""brown rice""","""chicken"""


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.arr.sum.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.arr.mean.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.arr.max.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.arr.min.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.arr.head.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.arr.tail.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.arr.first.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.arr.last.html

---
# Ejercicios

## Ejercicio 1
Dado el siguiente DataFrame, calcula cuántos productos tiene cada tienda.

In [ ]:
tiendas = pl.DataFrame({
    "tienda": ["Centro", "Norte", "Sur"],
    "productos": ["pan;leche;huevos", "arroz;frijoles", "pan;leche;arroz;café;azúcar"]
})
tiendas

tienda,productos
str,str
"""Centro""","""pan;leche;huevos"""
"""Norte""","""arroz;frijoles"""
"""Sur""","""pan;leche;arroz;café;azúcar"""


## Ejercicio 2
Dado el siguiente DataFrame, crear una nueva columna `"aprobados"` que contenga solo las notas mayores o iguales a 6. Luego agrega la columna `cantidad_aprobados` con el número de aprobados de cada estudiante.

In [ ]:
notas = pl.DataFrame({
    "estudiante": ["Ana", "Luis", "María"],
    "calificaciones": [[8, 5, 9, 3], [6, 7, 6], [4, 3, 5, 2, 8]]
})

## Ejercicio 3
Dado el siguiente DataFrame con dos columnas de listas, obtener todas las combinaciones de jugador con sus goles. Elimina los valores nulos del resultado.

In [ ]:
equipos = pl.DataFrame({
    "equipo": ["Tigres", "Leones", "Águilas"],
    "jugadores": [["Carlos", "Pedro"], ["Ana", "Lucía", "Marta"], ["Diego"]],
    "goles": [[2, 1], [3, 0, 2], [5]]
})
equipos

equipo,jugadores,goles
str,list[str],list[i64]
"""Tigres""","[""Carlos"", ""Pedro""]","[2, 1]"
"""Leones""","[""Ana"", ""Lucía"", ""Marta""]","[3, 0, 2]"
"""Águilas""","[""Diego""]",[5]


## Ejercicio 4
Crea un DataFrame con una columna `"ingredientes"` de tipo `pl.Array`. Luego usa el atributo `arr` para obtener el primer y último ingrediente de cada receta.

In [ ]:
recetas = pl.DataFrame(
    {
        "plato": ["Tacos", "Pasta", "Ensalada"],
        "ingredientes": [
            ["tortilla", "carne", "salsa"],
            ["espagueti", "tomate", "queso"],
            ["lechuga", "tomate", "aderezo"]
        ]
    },
    schema_overrides={"ingredientes": pl.Array(pl.String, shape=3)}
)
recetas

plato,ingredientes
str,"array[str, 3]"
"""Tacos""","[""tortilla"", ""carne"", ""salsa""]"
"""Pasta""","[""espagueti"", ""tomate"", ""queso""]"
"""Ensalada""","[""lechuga"", ""tomate"", ""aderezo""]"


## Ejercicio 5
Dado el siguiente DataFrame, agregar una columna `"descripcion"` que combine ciudad, país y continente separados por `" - "`. Además, crea una columna `"datos_lista"` que agrupe los tres valores en una lista.

In [ ]:
ciudades = pl.DataFrame({
    "ciudad": ["Lima", "Madrid", "Tokio"],
    "pais": ["Perú", "España", "Japón"],
    "continente": ["América", "Europa", "Asia"]
})
ciudades

ciudad,pais,continente
str,str,str
"""Lima""","""Perú""","""América"""
"""Madrid""","""España""","""Europa"""
"""Tokio""","""Japón""","""Asia"""
